# Does Female Representation in Governments affect GDP Growth Rate?

A study of whether female representation in government, both in parliament and at ministerial levels, is associated with a country's annual GDP growth rate.

## Section 1: Hypothesis, Analysis Goal

#### Hypothesis/analysis goal (Ask)
Clearly state the data science hypothesis / the analytical goal you pursue in your project. From your description, it should be clear which question you address, what type of analysis is needed, and what quality metric is targeted.

As nations across the world start to implement policies to improve gender equality on the governmental front, it would be interesting to see if this trend would affect economic metrics of countries. This study serves to find out if there is a correlation between governmental female representation and the economic growth of the nation. Governmental female representation will be measured by the proportion of females in Parliament, as well as the proportion of females in ministerial level positions in a particular year. Economic growth will then be measured by the GDP growth rate of the country that particular year, as compared to the previous year.


## Section 2: Data Sources and Requirements

#### Data source identification and exploration (Prepare)
Make a brief requirements analysis for the data and data sources that you require to reach your goal.

Requirement analysis. To tackle the problem, we need to have data about football matches that include when and where the match took place and what the result was. This data needs to be aligned with historical weather data at these times and locations. For accurate analysis, it is desirable to have sufficient data on matches for all different climatic conditions, requiring a broad coverage of geographies. It is sufficient to have one summary entry per match (no single events in matches) because the prediction aims at the final score. Climatic conditions vary seasonally so we do not need the weather at fine granularity (during the match time), day or even month may be enough.

To investigate this relationship, we need data about:

- GDP Growth Rates
- Female Representation in Parliament

**List of considered sources.** In the following, we outline several sources we considered and explored and justify the final source selection (marked with an *), in particular based on concepts discussed in class.

**Global Annual Gross Domestic Product (GDP) Growth Rates**
  * [World Bank GDP Growth Rates (DS1)](https://data.worldbank.org/indicator/NY.GDP.MKTP.KD.ZG)
   
**Female Representation in Parliament Data**
  * [Proportion of Seats Held by Women in National Parliaments (%) (DS2)](https://genderdata.worldbank.org/en/indicator/sg-gen-parl-zs)
  * [Proportion of Women in Ministerial Level Positions (%) (DS3)](https://genderdata.worldbank.org/en/indicator/sg-gen-mnst-zs)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 3. DS1: World Bank Annual GDP Growth Rates Exploration

**Description**

The World Bank dataset, downloaded on 29/01/2026, provides annual GDP growth percentages for 266 entities. While the raw data spans from 2005 to 2024, the temporal granularity was standardised to start from 2005 to ensure perfect alignment with the secondary "Women in Parliament" indicators. This prevents data sparsity issues and ensures a balanced panel for multivariate analysis.

To maintain statistical integrity, the numeric growth data was enriched with country metadata. This allowed for two critical preparation steps:

Removing Aggregates: Entities representing regional or global groups (e.g., "World", "High Income") were identified by their null geographic regions and systematically removed. This isolates 217 individual nations, preventing the "double-counting" of economic data that occurs when both a country and its parent economic group are included in the same average.

Income Group Classification: Each country was categorized by the World Bank's Income Group (High, Upper-Middle, Lower-Middle, and Low income). This enrichment facilitates a diagnostic analysis, allowing us to observe if the relationship between womenâ€™s leadership and GDP growth varies across different stages of economic development.

The indicator denotes the percentage change in GDP at market prices based on constant local currency. The following code imports the data, transforms it from a wide format to a longitudinal long format, and executes the enrichment and cleaning logic described above.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

gdp_file = 'sample_data/API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_26.csv'
meta_file = 'sample_data/Metadata_Country_API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_26.csv'

# 1. Load the main GDP data
df_gdp_wide = pd.read_csv(gdp_file, skiprows=4)

# 2. Clean the wide data
df_gdp_wide = df_gdp_wide.loc[:, ~df_gdp_wide.columns.str.contains('^Unnamed')]

# 3. Transform: Melt from 'Wide' to 'Long'
df_gdp_long = pd.melt(
    df_gdp_wide,
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    var_name='Year',
    value_name='GDP_Growth'
)

# Convert Year to numeric
df_gdp_long['Year'] = pd.to_numeric(df_gdp_long['Year'], errors='coerce')

# 4. Load and Join Metadata
df_metadata = pd.read_csv(meta_file)
df_enriched = pd.merge(
    df_gdp_long,
    df_metadata[['Country Code', 'Region', 'IncomeGroup']],
    on='Country Code',
    how='left'
)

# 5. Final Cleaning & Alignment (2005 onwards)
df_countries = df_enriched.dropna(subset=['Region'])
df_final = df_countries[df_countries['Year'] >= 2005].reset_index(drop=True)

# 6. Display the numbers with the "..." format
# Simply call the dataframe name at the end of the cell
df_final[['Country Name', 'Year', 'GDP_Growth', 'Region', 'IncomeGroup']]

FileNotFoundError: [Errno 2] No such file or directory: 'sample_data/API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_26.csv'

Through our data cleaning, we have differentiated sovereign nations from regional aggregates to ensure statistical accuracy and avoid double-counting. In terms of classification, the 'Region' and 'IncomeGroup' metadata are provided, which offer sufficient granularity to control for geographic and economic differences in our analysis. Similarly, the annual granularity is appropriate for identifying the longitudinal relationship between political shifts and economic outcomes. For our analytics tasks, we need to ensure that we sufficiently cover a variety of economic contexts, specifically ensuring representation across Low and Middle Income nations to mitigate the reporting bias often found in high-income datasets.

#### Profiling completeness by income group (2023)

In [ ]:
# Profiling: Count non-null values per Income Group for the year 2023
profile_2023 = df_enriched[df_enriched['Year'] == 2023].groupby('IncomeGroup')['GDP_Growth'].count()
print("Data Completeness for 2023 by Income Group:\n", profile_2023)

To ensure the datasetâ€™s reliability before conducting further analysis, we profiled the results for 2023. We specifically selected 2023 as our baseline year rather than 2024 because it currently offers the highest data completeness (approximately 93% coverage). By grouping GDP growth by income level, we verify the successful integration of our primary and metadata datasets and assess the representativeness of our sample, identifying any potential biases.

#### Exploring Countries Covered in the Dataset

In [ ]:
# List the distinct countries in the dataset to verify granularity
distinct_countries = df_final['Country Name'].unique()
print(f"Total Unique Nations: {len(distinct_countries)}")
print(distinct_countries)

While the program above provides high granularity for sovereign nations and autonomous territories, it lacks sub-national granularity for federal states. For example, the economic nuances of self-governing US states or German BundeslÃ¤nder are aggregated into a single national figure. This represents a limitation for our later comparison with female representation as different governance systems distribute legislative power differently.

#### Exploring Income Group Representations

In [ ]:
# List the distinct Income Groups to verify categorical granularity
distinct_income_groups = df_final['IncomeGroup'].unique()
print(distinct_income_groups)

A wide range of economic development levels were covered, from Low Income to High Income. Thus, the dataset is representative of national economies operating at various levels of industrialisation. This alleviates the risk of focusing only on bigger economies and allows our analysis to generalise across different global contexts.

#### Data quality considerations

We highlight the following quality criteria that we assessed (among others) to determine the suitability of the dataset for our task:

- *Completeness*: The dataset satisfies our technical requirements for longitudinal analysis as it offers more than 27 years of annual GDP growth globally. Our profiling also confirmed that the data provides sufficient breadth (covering sovereign nations, self-governing states, and autonomous territories) and depth (stable reporting from 1997 to 2024, with a high reporting rate of 93% in 2023). These factors ensure that our findings will be statistically representative across diverse economic contexts.

- *Interpretability*: The interpretability of this dataset is exceptionally high. The indicator (annual % GDP growth) has clear, standardised semantics based on constant local currency, which minimises the risk of misinterpretation. Furthermore, the inclusion of IncomeGroup and Region metadata provides clear categorical attributes for diagnostic analysis, allowing us to segment results by economic development levels without ambiguity.

- *Reputation*: Unlike scraped or community-sourced data, this dataset has a high reputation as it is maintained by the World Bank Group. The provenance is also well-documented, with established methodologies for data collection from national statistical bureaus.

Although the aggregation of sub-national data in federal systems could introduce minor biases, the alignment of this dataset with our gender-based indicators makes it the most suitable choice for our research task.

#### Conclusion for World Bank Annual GDP Growth Rates Dataset

We find the dataset suited for our analysis with the following caveats, which we will take into account during data preparation and analysis:
- Locations may exhibit an overrepresentation of High Income economies, which could possibly be countered by strategic sampling during the analysis.
- The aggregation of sub-national data into national averages may introduce minor biases. In federal systems where economic policies are decentralised, national-level representation may not fully capture the leadership dynamics driving growth at the state level. While cross-validating with sub-national datasets is an option to mitigate this, it is not further considered within the scope of this project.

## 4. DS2: Proportion of seats held by women in national parliaments (%)

**Description.** We downloaded the CSV file from Kaggle on 30/01/2026 from the World Bank Group website. The data is organised year by year, with 4 columns of Economy Code, Country Name, Proportion of Seats Held by Women in National Parliaments, and Year. All columns are necessary and will be used in our analysis. The World Bank Group defines the proportion of seats held by women in national parliaments is the number of seats held by women members in single or lower chambers of national parliaments, expressed as a percentage of all occupied seats; it is derived by dividing the total number of seats occupied by women by the total number of seats in parliament.

In [3]:
# prompt: list column labels from the CSV file sample_data/matches_1930_2022.csv

# Read the CSV file into a DataFrame
table = pd.read_csv('sample_data/2024.csv')

# Get the column labels (header)
column_labels = table.columns.tolist()

# Print the column labels
print(column_labels)

['Economy', 'Year', 'Economy Code', 'Proportion of seats held by women in national parliaments (%)']


#### Profiling geographic coverage

We perform some data profiling on the data to better understand the countries it covers.

In [4]:
# Showing the number of unique countries
unique_entities = table['Economy'].unique()
print(unique_entities)
print("Number of Total Entities: "+ str(len(unique_entities)))

<StringArray>
[          'Afghanistan',               'Albania',               'Algeria',
        'American Samoa',               'Andorra',                'Angola',
   'Antigua and Barbuda',             'Argentina',               'Armenia',
                 'Aruba',
 ...
               'Uruguay',            'Uzbekistan',               'Vanuatu',
         'Venezuela, RB',               'Vietnam', 'Virgin Islands (U.S.)',
    'West Bank and Gaza',           'Yemen, Rep.',                'Zambia',
              'Zimbabwe']
Length: 217, dtype: str
Number of Total Entities: 217


Looking at the above statistics and value distributions, we see that the countries that this dataset covers is comprehensive, a total of 217 economies. This covers a broad geographical scope, ensuring our analysis captures diverse contexts, from developing nations to emerging markets and developing economies, making our findings more generalizable across all types of economies.

However, we do have to take not that not all economies are sovereign nations under the UN, since there is only a total of 195 registered. There are some terrorities and dependencies:

- British Virgin Islands, American Samoa, Guam, Puerto Rico, U.S. Virgin Islands (U.S. territories)
- Bermuda, Cayman Islands, Gibraltar (British territories) This might count in some countries being double counted and needs to be dealt with during data preprocessing.

Also, the temporal depth of this dataset makes it a suitable choice as well, spanning 18 years from 1997 to 2024. The coverage across time and space makes it comprehensive enough for our analysis.

#### Quality considerations

Using the criteria established above:
- **Completeness**: The dataset satisfies our technical requirements for longitudinal analysis as it offers 18 years of annual GDP growth globally. Our profiling also confirmed that the data provides sufficient breadth of countries in terms of economy type, geographical location and other aspects.

- **Interpretability**: The interpretability of this dataset is very high. The indicator (Proportion of Seats Held by Females in National Parliaments) is clear and requires no further clarification

- **Reputation**: This dataset was found in the World Bank Group website and has a high reputation, as well as good documentation detailing how key metrics were calculated.

#### Conclusions for the World Bank Group dataset

By itself, the World Bank Group dataset is ideal for our analytical goal, mainly due to its coverage of time periods and geographies. We will therefore use this dataset for our project.

## 5. DS3: Proportion of women in ministerial level positions (%)
**Description.** Women in ministerial level positions is the proportion of women in ministerial or equivalent positions (including deputy prime ministers) in the government. Prime Ministers/Heads of Government are included when they hold ministerial portfolios. Vice-Presidents and heads of governmental or public agencies are excluded.

The dataset is retrieved from World Bank Group consisting of the following data:

- **indicator name**:plain-English description of what is being measured, which in this case is the percentage of women in ministeral position.

- **indicator code**: unique ID (usually a string of letters and numbers) used by database systems to identify the percentage of women in ministeral position metric.

- **country name**: full name of the country, territory, or region.

- **country code**:3-letter abbreviation (ISO code) for the country, territory, or region.

- **Year**: calendar year the data was collected.

- **value**: actual percentage of women in ministeral position.

**Note that country include aggregates in terms of region (eg. Africa Western and Central) which may result in double counting**

**Viewing.** We take a look at this dataset after removing irrelevant columns such as indicator name, indicator code and disaggregation (which has no values logged in CSV).

In [ ]:
# Read the CSV file into a DataFrame
df_proportion = pd.read_csv('sample_data/Proportion of women in ministerial level positions (%).csv')

# Remove the columns not needed
cols_to_remove = ['Indicator Name', 'Indicator Code', 'Disaggregation']
df_proportion = df_proportion.drop(columns=cols_to_remove)

# Convert 'Year' to datetime
df_proportion['Year'] = pd.to_datetime(df_proportion['Year'], format='%Y')

# Look at the cleaned data
df_proportion.head(50)

#### Profiling date coverage

Verify the date range of the dataset to so that it can be altered later on to correctly overlap with the date range of Annual GDP and Proportion of Seats in Parliament datasets.

In [ ]:
# Check the earliest and latest dates in the dataset
min_date = df_proportion['Year'].min()
max_date = df_proportion['Year'].max()

print(f"Data starts from: {min_date}")
print(f"Data ends at: {max_date}")

Data starts from: 2005-01-01 00:00:00
Data ends at: 2024-01-01 00:00:00
Profiling geographical coverage

Evaluate the total number of unique countries and regions to identify potential double-counting caused by the inclusion of aggregate regional data which will be delt with during data cleaning later.

In [ ]:
# Count the number of unique Country Codes
unique_count = df_proportion['Country Code'].nunique()

# Get a list of all unique Country Names to see the Regions vs Countries
unique_entities = df_proportion['Country Name'].unique()

print(f"Total unique entities (Countries + Regions): {unique_count}")
print("-" * 30)
print("List of entities:")
print(unique_entities)

The exploration shows 245 number of unique countries and regions. Given that there are 195 independent countries in the world, this indicates the presence of 50 regional data which overlaps with the country data, for example:
- the region Africa Eastern and Southern
- region includes other African countries

This can result in double counting and skewing of results.

**Profiling of completeness of data**

Validate that a percentage data exists for each country/region for each year

In [ ]:
# 1. Filter for the specific timeframe (2005 to 2024)
mask = (df_proportion['Year'] >= '2005-01-01') & (df_proportion['Year'] <= '2024-01-01')
df_filtered = df_proportion.loc[mask]

# 2. Group by Country Code and count the entries in the 'Value' column
# .count() only counts non-null (not empty) values
data_counts = df_filtered.groupby('Country Code')['Value'].count().reset_index()

# 3. Rename columns for clarity
data_counts.columns = ['Country Code', 'Data_Points_Count']

# 4. Sort by the count so you can see who has the most/least data
data_counts = data_counts.sort_values(by='Data_Points_Count', ascending=False)

# Display the result
print(data_counts.head(20)) # Show top 20

Data Frequency Limitation: While the study covers a 20-year period (2005â€“2024), the dataset contains only 13 observations per entity, confirming that reporting is non-annual.

Data is recorded for select years: 2024, 2023, 2022, 2020, 2019, 2018, 2016, 2015, 2014, 2012, 2010, 2008, and 2005. This irregular reporting schedule presents a significant challenge for downstream analysis, as the lack of a continuous time series prevents standard Year-over-Year (YoY) growth comparisons.

In [ ]:
# 1. Filter for the specific timeframe (2005 to 2024)
mask = (df_proportion['Year'] >= '2005-01-01') & (df_proportion['Year'] <= '2024-01-01')
df_filtered = df_proportion.loc[mask]

# 2. Group by Country Code and count the non-null entries
data_counts = df_filtered.groupby('Country Code')['Value'].count().reset_index()

# 3. Rename columns
data_counts.columns = ['Country Code', 'Data_Points_Count']

# --- NEW STEP: FILTER AND COUNT ---

# 4. Filter for ISO codes where the count is NOT 13
not_13_df = data_counts[data_counts['Data_Points_Count'] != 13]

# 5. Get the total count of these ISO codes
total_not_13 = len(not_13_df)

# Display results
print(f"Number of ISO codes that do not have 13 data points: {total_not_13}")
print("-" * 30)
print("Top rows of entities without 13 data points:")
display(not_13_df.head())

The above are the regions/countries that have missing datasets which need to be noted when analysing later on.

##### Quality considerations

**Completeness**: The dataset satisfies our technical requirements for longitudinal analysis by providing a broad historical range from 2005 to 2024. While the ministerial data follows an irregular reporting schedule (13 observations over 20 years), it offers sufficient breadth across geographical regions and income levels to facilitate a comparative global study.

**Interpretability**: The interpretability of this dataset is very high. The primary indicatorâ€”Proportion of Women in Ministerial Level Positions (%)â€”is clearly defined as the percentage of cabinet members or ministers who are female, requiring no further technical clarification for analysis.

**Reputation**: This dataset is sourced from the World Bank Gender Data Portal and carries a high reputation for accuracy. It is supported by robust documentation and standardized "Country Codes" (ISO), ensuring that the methodology for data collection and aggregation is transparent and reliable for academic or professional research.

##### Conclusion for the World Bank Group Data

We will take this dataset into account for more accurate reflection of women's involvement in politics with careful consideration when cleaning the data